# Visualizing a static structure in XR with NanoVer

In this example we will visualize LSD bound into 5-HT2B receptor with MDAnalysis (without dynamics) and configure the server visualizations shared between users in NanoVer iMD-XR client.

## Structure setup

Just as in the [trajectory playback notebook](../0.%20basics/trajectory_playback.ipynb), we'll user MDAnalysis to load a PDB as a NanoVer compatible simulation but this time just the topology without additional trajectory information.

In [1]:
TOPOLOGY_PATH = "../systems/serotonin_receptor.pdb"

In [2]:
from MDAnalysis import Universe

# we need MDAnalysis to fill in the missing bonds
universe = Universe(
    TOPOLOGY_PATH,
    vdwradii={'Na': 0, 'Cl': 0},
    to_guess=("bonds",),
)

## NanoVer simulation and server setup

Next we wrap the universe in and host it with NanoVer in the usual way:

In [3]:
from nanover.app import OmniRunner
from nanover.mdanalysis import UniverseSimulation

structure_sim = UniverseSimulation.from_universe(universe)

imd_runner = OmniRunner.with_basic_server(structure_sim, port=0, name="visualize static lsd example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "visualize static lsd example" (ws://localhost:63700), discoverable on all interfaces on port 54545
Available simulations:
[0]: "../systems/serotonin_receptor.pdb"
Switched to [0]: "../systems/serotonin_receptor.pdb"
Switched to [0]: "../systems/serotonin_receptor.pdb"


If you connect to the server from VR, you'll see something like this:

<img src="images/lsd_ball_and_stick.png" alt="LSD Ball and Stick" style="width: 500px;"/>


## Visualization configuration

The default ball and stick representation is fine, but we can customize it right here from the notebook. We'll use a utility functions for modifying the selections.

In [4]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

Before we start, we'll define some custom colors for the elements:

In [5]:
cpk_colours = {
    'N': 'blue',
    'P': '#dca523',
    'C': '#c0c0c0',
    'O': '#fc1c03',
    'S': '#e9ce16'
}

We'll also define a function that copies color gradients from matplotlib:

In [6]:
import matplotlib

def get_matplotlib_gradient(name: str):
    cmap = matplotlib.colormaps.get_cmap(name)
    return [list(cmap(x/7)) for x in range(0, 8)]

Now we'll get started by hiding the `root` selection, which always contains every atom of the system. Then we'll introduce new selections for each individual part of the system that we want to see.

In [7]:
utilities.selections.update_selection("root", hide=True)

We'll make use of MDAnalysis' functionality to query the atoms of interest: the receptor protein (minus hydrogens), the lipids (residue DPPC), and the ligands (everything else):

In [8]:
PROTEIN_ATOMS = universe.select_atoms("protein and not type H")
LIPID_ATOMS = universe.select_atoms("resname DPPC")
LIGAND_ATOMS = universe.select_atoms("not protein and not resname DPPC")

Now, we'll render the protein using our tetrahedral spline renderer, coloured with the lovely viridis colour scheme. 
We do this by editting the `renderer` settings, which is just a dictionary of settings.

In [9]:
utilities.selections.update_selection(
    "protein",
    particle_ids=PROTEIN_ATOMS.indices,
    renderer={
        'sequence': 'polypeptide',
        'color': {
            'type': 'residue index in entity',
            'gradient': get_matplotlib_gradient('viridis')
        },
        'render': 'geometric spline'
    },
    interaction_method="none",
)

Alternatively, you can colour by secondary structure:

In [10]:
utilities.selections.modify_selection("protein", renderer="cartoon")

We'll add the ligands back in with CPK liquorice:

In [11]:
utilities.selections.update_selection(
    "ligands",
    particle_ids=LIGAND_ATOMS.indices,
    renderer={
        'color': {
            'type': 'cpk',
            'scheme': cpk_colours,
        },
        'scale': 0.1,
        'render': 'liquorice'
    },
    interaction_method="none",
)

And the lipids with a scaled down liquorice:

In [12]:
utilities.selections.update_selection(
    "lipids",
    particle_ids=LIPID_ATOMS.indices,
    renderer={
        'color': {
            'type': 'cpk',
            'scheme': cpk_colours,
        },
        'scale': 0.01,
        'render': 'liquorice'
    },
    interaction_method="none",
)

You may find the lipids are a bit much - you can easily hide them. Just uncomment and run this line:

In [13]:
#utilities.selections.modify_selection("lipids", hide=True)

## Visualization based on distance

Finally, we'll render the sidechains near ligand:

In [14]:
NEARBY_ATOMS = universe.select_atoms("(protein and (not backbone or name CA)) and same resid as around 4 resname 7LD")

In [15]:
utilities.selections.update_selection(
    "nearby",
    particle_ids=NEARBY_ATOMS.indices,
    renderer={
        'color': {
            'type': 'residue index in entity',
            'gradient': get_matplotlib_gradient('viridis')
        },
        'scale': 0.05,
        'render': 'cycles'
    },
    interaction_method="none",
)

And try an alternate rendering:

In [16]:
utilities.selections.modify_selection(
    "nearby",
    renderer={
        'color': {
            'type': 'cpk',
            'scheme': cpk_colours,
        },
        'scale': 0.05,
        'render': 'liquorice'
    },
)

Depending on which cells you ran, you'll have something that looks like this, much better! (maybe without the lipids)

<img src="images/lsd_nanover.png" style="width: 400px;  display: inline-block; vertical-align: top;">
<img src="images/lsd_nanover_option_2.png" style="width: 305px;  display: inline-block; vertical-align: top;">